# K-mer Embedding CNN for Binary Classification

This notebook experiments with k-mer embeddings for DNA sequence classification.
K-mer encoding has been shown to achieve 93%+ accuracy in DNA sequence classification tasks.

## Key Features:
- Uses k-mer tokenization instead of one-hot encoding
- Embedding layer to learn k-mer representations
- CNN architecture optimized for k-mer sequences
- Comparison with traditional one-hot encoding approaches


In [9]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from typing import List, Optional, Tuple
import random
from pathlib import Path

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


## K-mer Utility Functions


In [10]:
# K-mer utility functions
SPECIALS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]"]

def build_kmer_vocab(k: int):
    """Build vocabulary for k-mers of length k."""
    bases = ["A", "C", "G", "T"]
    kmers = ["".join(p) for p in product(bases, repeat=k)]
    vocab = SPECIALS + kmers
    stoi = {t: i for i, t in enumerate(vocab)}
    itos = {i: t for t, i in stoi.items()}
    return vocab, stoi, itos

def seq_to_kmers(seq: str, k: int) -> List[str]:
    """Convert DNA sequence to k-mer tokens."""
    seq = seq.upper()
    toks: List[str] = []
    for i in range(len(seq) - k + 1):
        kmer = seq[i : i + k]
        if any(c not in "ACGT" for c in kmer):
            toks.append("[UNK]")
        else:
            toks.append(kmer)
    return ["[CLS]"] + toks + ["[SEP]"]

# Test k-mer functions
k = 6
vocab, stoi, itos = build_kmer_vocab(k)
print(f"K-mer size: {k}")
print(f"Vocabulary size: {len(vocab)}")
print(f"First 10 k-mers: {vocab[4:14]}")

# Test sequence tokenization
test_seq = "ATCGATCGATCG"
kmers = seq_to_kmers(test_seq, k)
print(f"\nTest sequence: {test_seq}")
print(f"K-mers: {kmers}")


K-mer size: 6
Vocabulary size: 4100
First 10 k-mers: ['AAAAAA', 'AAAAAC', 'AAAAAG', 'AAAAAT', 'AAAACA', 'AAAACC', 'AAAACG', 'AAAACT', 'AAAAGA', 'AAAAGC']

Test sequence: ATCGATCGATCG
K-mers: ['[CLS]', 'ATCGAT', 'TCGATC', 'CGATCG', 'GATCGA', 'ATCGAT', 'TCGATC', 'CGATCG', '[SEP]']


## Dataset Class for K-mer Embeddings


In [11]:
class KmerBinaryClassificationDataset(Dataset):
    """Dataset for binary classification using k-mer embeddings."""
    def __init__(self, data, k=6, max_length=None):
        self.data = data.reset_index(drop=True)
        self.k = k
        self.vocab, self.stoi, self.itos = build_kmer_vocab(k)
        self.vocab_size = len(self.vocab)
        
        # Calculate max sequence length in k-mers if not provided
        if max_length is None:
            max_kmers = max(len(seq_to_kmers(seq, k)) for seq in self.data['ProSeq'])
            self.max_length = min(max_kmers, 512)  # Cap at reasonable length
        else:
            self.max_length = max_length
            
    def __len__(self): 
        return len(self.data)
    
    def __getitem__(self, idx):
        sequence = self.data.iloc[idx]['ProSeq']
        target = self.data.iloc[idx]['binary_classification']
        
        # Convert sequence to k-mer tokens
        kmers = seq_to_kmers(sequence, self.k)
        
        # Convert to token IDs with padding/truncation
        pad_id = self.stoi["[PAD]"]
        unk_id = self.stoi["[UNK]"]
        
        ids = [self.stoi.get(kmer, unk_id) for kmer in kmers[:self.max_length]]
        attention_mask = [1] * len(ids)
        
        # Pad if necessary
        if len(ids) < self.max_length:
            pad_n = self.max_length - len(ids)
            ids += [pad_id] * pad_n
            attention_mask += [0] * pad_n
            
        return torch.tensor(ids, dtype=torch.long), torch.tensor(attention_mask, dtype=torch.bool), target


## K-mer Embedding CNN Model


In [12]:
class KmerEmbeddingCNN(nn.Module):
    """CNN architecture using k-mer embeddings for binary classification."""
    def __init__(self, vocab_size, embedding_dim=128, num_conv_layers=3, 
                 conv_channels=[32, 64, 128], kernel_sizes=[3, 5, 7], pool_sizes=[2, 2, 2],
                 num_fc_layers=1, fc_sizes=[64], dropout_rate=0.3, 
                 use_batch_norm=True, activation='relu', pooling_type='max'):
        super(KmerEmbeddingCNN, self).__init__()
        self.num_conv_layers = num_conv_layers
        self.num_fc_layers = num_fc_layers
        self.dropout_rate = dropout_rate
        
        # Activation function
        activation_map = {
            'relu': nn.ReLU, 
            'leaky_relu': nn.LeakyReLU, 
            'gelu': nn.GELU, 
            'swish': nn.SiLU, 
            'elu': nn.ELU
        }
        self.activation = activation_map[activation]()
        
        # K-mer embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)  # 0 is [PAD] token
        self.embedding_dropout = nn.Dropout(dropout_rate)
        
        # Convolutional layers
        self.conv_layers = nn.ModuleList()
        self.bn_layers = nn.ModuleList()
        self.pool_layers = nn.ModuleList()
        
        in_channels = embedding_dim
        for i in range(num_conv_layers):
            out_channels = conv_channels[i]
            self.conv_layers.append(nn.Conv1d(in_channels, out_channels, kernel_sizes[i], padding=kernel_sizes[i]//2))
            self.bn_layers.append(nn.BatchNorm1d(out_channels) if use_batch_norm else nn.Identity())
            self.pool_layers.append(nn.MaxPool1d(pool_sizes[i]))
            in_channels = out_channels
            
        # Global pooling
        self.pooling_type = pooling_type
        if pooling_type == 'avg': 
            self.global_pool = nn.AdaptiveAvgPool1d(1)
        elif pooling_type == 'max': 
            self.global_pool = nn.AdaptiveMaxPool1d(1)
        else:  # 'both'
            self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
            self.global_max_pool = nn.AdaptiveMaxPool1d(1)
            in_channels *= 2
            
        # Fully connected layers
        self.fc_layers = nn.ModuleList()
        self.fc_bn_layers = nn.ModuleList()
        
        if num_fc_layers > 0:
            fc_input_size = in_channels
            for i in range(num_fc_layers):
                fc_output_size = fc_sizes[i]
                self.fc_layers.append(nn.Linear(fc_input_size, fc_output_size))
                self.fc_bn_layers.append(nn.BatchNorm1d(fc_output_size) if use_batch_norm else nn.Identity())
                fc_input_size = fc_output_size
            self.output_layer = nn.Linear(fc_input_size, 1)
        else: 
            self.output_layer = nn.Linear(in_channels, 1)
            
        self._initialize_weights()
        
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv1d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: 
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d): 
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0, std=0.1)
                
    def forward(self, input_ids, attention_mask=None):
        # Embed k-mer tokens: [batch_size, seq_len] -> [batch_size, seq_len, embedding_dim]
        x = self.embedding(input_ids)
        x = self.embedding_dropout(x)
        
        # Apply attention mask if provided
        if attention_mask is not None:
            x = x * attention_mask.unsqueeze(-1).float()
        
        # Transpose for Conv1d: [batch_size, seq_len, embedding_dim] -> [batch_size, embedding_dim, seq_len]
        x = x.transpose(1, 2)
        
        # Apply convolutional layers
        for i in range(self.num_conv_layers):
            x = self.activation(self.bn_layers[i](self.conv_layers[i](x)))
            x = self.pool_layers[i](x)
            x = F.dropout(x, p=self.dropout_rate, training=self.training)
            
        # Global pooling
        if self.pooling_type == 'both': 
            x = torch.cat([self.global_avg_pool(x), self.global_max_pool(x)], dim=1).squeeze(-1)
        else: 
            x = self.global_pool(x).squeeze(-1)
            
        # Fully connected layers
        for i in range(self.num_fc_layers):
            x = self.activation(self.fc_bn_layers[i](self.fc_layers[i](x)))
            x = F.dropout(x, p=self.dropout_rate, training=self.training)
            
        return torch.sigmoid(self.output_layer(x))


## Load and Prepare Data


In [13]:
# Load data
data_path = '../../../data/processed/ProSeq_binary_classification.csv'
print(f"Loading data from: {data_path}")

data_binary = pd.read_csv(data_path)
data_filtered = data_binary[data_binary['ProSeq'].str.len() >= 600].copy()
print(f"Dataset size: {len(data_filtered)}")

# Check class distribution
class_counts = data_filtered['binary_classification'].value_counts()
print(f"\nClass distribution:")
print(class_counts)
print(f"Class balance: {class_counts.min() / class_counts.max():.3f}")

# Calculate class weights
class_weights = compute_class_weight('balanced', 
                                   classes=np.unique(data_filtered['binary_classification']),
                                   y=data_filtered['binary_classification'])
print(f"Class weights: {class_weights}")


Loading data from: ../../../data/processed/ProSeq_binary_classification.csv
Dataset size: 8302

Class distribution:
binary_classification
0    5217
1    3085
Name: count, dtype: int64
Class balance: 0.591
Class weights: [0.79566801 1.34554295]


In [14]:
# Split data
train_val_data, test_data = train_test_split(
    data_filtered, test_size=0.2, random_state=SEED, 
    stratify=data_filtered['binary_classification']
)
train_data, val_data = train_test_split(
    train_val_data, test_size=0.2, random_state=SEED, 
    stratify=train_val_data['binary_classification']
)

print(f"Data splits:")
print(f"Train: {len(train_data)}")
print(f"Validation: {len(val_data)}")
print(f"Test: {len(test_data)}")


Data splits:
Train: 5312
Validation: 1329
Test: 1661


## Create K-mer Datasets and Data Loaders


In [15]:
# Create k-mer datasets
K_MER_SIZE = 6
print(f"Using k-mer size: {K_MER_SIZE}")

train_dataset = KmerBinaryClassificationDataset(train_data, k=K_MER_SIZE)
val_dataset = KmerBinaryClassificationDataset(val_data, k=K_MER_SIZE)
test_dataset = KmerBinaryClassificationDataset(test_data, k=K_MER_SIZE)

print(f"Vocabulary size: {train_dataset.vocab_size}")
print(f"Max sequence length (k-mers): {train_dataset.max_length}")

# Test dataset
sample_input_ids, sample_mask, sample_target = train_dataset[0]
print(f"\nSample shapes:")
print(f"Input IDs: {sample_input_ids.shape}")
print(f"Attention mask: {sample_mask.shape}")
print(f"Target: {sample_target}")

# Show some k-mer tokens
non_pad_tokens = sample_input_ids[sample_mask]
print(f"\nFirst 10 k-mer tokens: {[train_dataset.itos[int(token)] for token in non_pad_tokens[:10]]}")


Using k-mer size: 6
Vocabulary size: 4100
Max sequence length (k-mers): 512

Sample shapes:
Input IDs: torch.Size([512])
Attention mask: torch.Size([512])
Target: 0

First 10 k-mer tokens: ['[CLS]', 'TCTGTC', 'CTGTCG', 'TGTCGC', 'GTCGCC', 'TCGCCA', 'CGCCAG', 'GCCAGG', 'CCAGGC', 'CAGGCC']


In [16]:
# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Data loaders created:")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Test a batch
for batch_input_ids, batch_mask, batch_targets in train_loader:
    print(f"\nBatch shapes:")
    print(f"Input IDs: {batch_input_ids.shape}")
    print(f"Attention mask: {batch_mask.shape}")
    print(f"Targets: {batch_targets.shape}")
    break


Data loaders created:
Train batches: 166
Val batches: 42
Test batches: 52

Batch shapes:
Input IDs: torch.Size([32, 512])
Attention mask: torch.Size([32, 512])
Targets: torch.Size([32])


## Create and Test Model


In [17]:
# Create model
model = KmerEmbeddingCNN(
    vocab_size=train_dataset.vocab_size,
    embedding_dim=128,
    num_conv_layers=3,
    conv_channels=[32, 64, 128],
    kernel_sizes=[3, 5, 7],
    pool_sizes=[2, 2, 2],
    num_fc_layers=1,
    fc_sizes=[64],
    dropout_rate=0.3,
    use_batch_norm=True,
    activation='relu',
    pooling_type='max'
).to(device)

print(f"Model created on device: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Test forward pass
model.eval()
with torch.no_grad():
    test_input_ids = batch_input_ids[:4].to(device)
    test_mask = batch_mask[:4].to(device)
    test_output = model(test_input_ids, test_mask)
    print(f"\nTest forward pass:")
    print(f"Input shape: {test_input_ids.shape}")
    print(f"Output shape: {test_output.shape}")
    print(f"Output values: {test_output.squeeze().cpu().numpy()}")


Model created on device: cpu
Total parameters: 613,793
Trainable parameters: 613,793

Test forward pass:
Input shape: torch.Size([4, 512])
Output shape: torch.Size([4, 1])
Output values: [0.6049622  0.57395905 0.59398466 0.4218902 ]


## Training Functions


In [18]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for input_ids, attention_mask, targets in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        targets = targets.float().to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        outputs = outputs.squeeze()
        
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_probs = []
    all_targets = []
    num_batches = 0
    
    with torch.no_grad():
        for input_ids, attention_mask, targets in val_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            targets = targets.float().to(device)
            
            outputs = model(input_ids, attention_mask)
            outputs = outputs.squeeze()
            
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            num_batches += 1
            
            # Collect predictions
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)
            targets_np = targets.cpu().numpy()
            
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_targets.extend(targets_np)
    
    avg_loss = total_loss / num_batches
    accuracy = accuracy_score(all_targets, all_preds)
    auc = roc_auc_score(all_targets, all_probs)
    
    return avg_loss, accuracy, auc, all_targets, all_preds, all_probs


## Train the Model


In [19]:
# Training setup
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Training parameters
num_epochs = 50
patience = 10
best_val_auc = 0.0
patience_counter = 0
best_model_state = None

# History tracking
history = {
    'train_losses': [],
    'val_losses': [],
    'val_accuracies': [],
    'val_aucs': []
}

print(f"Starting training for {num_epochs} epochs...")
print(f"Early stopping patience: {patience}")
print("-" * 80)


Starting training for 50 epochs...
Early stopping patience: 10
--------------------------------------------------------------------------------


In [21]:
# Training loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} of {num_epochs}")
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc, val_auc, _, _, _ = validate_epoch(model, val_loader, criterion, device)
    
    # Update history
    history['train_losses'].append(train_loss)
    history['val_losses'].append(val_loss)
    history['val_accuracies'].append(val_acc)
    history['val_aucs'].append(val_auc)
    
    # Early stopping check
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
    
    # Print progress
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{num_epochs} - '
              f'Train Loss: {train_loss:.4f}, '
              f'Val Loss: {val_loss:.4f}, '
              f'Val Acc: {val_acc:.4f}, '
              f'Val AUC: {val_auc:.4f} '
              f'(Best: {best_val_auc:.4f})')
    
    # Early stopping
    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'\nLoaded best model with validation AUC: {best_val_auc:.4f}')

print("\nTraining completed!")


Epoch 1 of 50
Epoch   1/50 - Train Loss: 3.0567, Val Loss: 46.4483, Val Acc: 0.3717, Val AUC: 0.5225 (Best: 0.5225)
Epoch 2 of 50
Epoch 3 of 50
Epoch 4 of 50
Epoch 5 of 50
Epoch   5/50 - Train Loss: 1.5543, Val Loss: 8.4298, Val Acc: 0.3717, Val AUC: 0.4822 (Best: 0.5225)
Epoch 6 of 50
Epoch 7 of 50
Epoch 8 of 50
Epoch 9 of 50
Epoch 10 of 50
Epoch  10/50 - Train Loss: 0.9957, Val Loss: 4.8766, Val Acc: 0.3717, Val AUC: 0.5098 (Best: 0.5225)
Epoch 11 of 50

Early stopping at epoch 11

Loaded best model with validation AUC: 0.5225

Training completed!


## Plot Training Curves


In [ ]:
# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

epochs = range(1, len(history['train_losses']) + 1)

# Training and validation loss
axes[0, 0].plot(epochs, history['train_losses'], 'b-', label='Training Loss')
axes[0, 0].plot(epochs, history['val_losses'], 'r-', label='Validation Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Validation accuracy
axes[0, 1].plot(epochs, history['val_accuracies'], 'g-', label='Validation Accuracy')
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Validation AUC
axes[1, 0].plot(epochs, history['val_aucs'], 'm-', label='Validation AUC')
axes[1, 0].set_title('Validation AUC')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUC')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Overfitting indicator
loss_diff = [val - train for val, train in zip(history['val_losses'], history['train_losses'])]
axes[1, 1].plot(epochs, loss_diff, 'orange', label='Val Loss - Train Loss')
axes[1, 1].set_title('Overfitting Indicator')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss Difference')
axes[1, 1].legend()
axes[1, 1].grid(True)
axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

print(f"Final training loss: {history['train_losses'][-1]:.4f}")
print(f"Final validation loss: {history['val_losses'][-1]:.4f}")
print(f"Best validation AUC: {best_val_auc:.4f}")
print(f"Final validation accuracy: {history['val_accuracies'][-1]:.4f}")


## Evaluate on Test Set


In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_acc, test_auc, test_targets, test_preds, test_probs = validate_epoch(
    model, test_loader, criterion, device
)

print(f"\nTest Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}")
print(f"Test F1: {f1_score(test_targets, test_preds):.4f}")

print(f"\nClassification Report:")
print(classification_report(test_targets, test_preds))


In [ ]:
# Plot confusion matrix
cm = confusion_matrix(test_targets, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Class 0', 'Class 1'], 
            yticklabels=['Class 0', 'Class 1'])
plt.title(f'K-mer CNN Confusion Matrix\nAccuracy: {test_acc:.4f}, AUC: {test_auc:.4f}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print(f"Confusion Matrix:")
print(f"True Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")


## Model Analysis and Comparison


In [ ]:
# Model parameter analysis
total_params = sum(p.numel() for p in model.parameters())
embedding_params = model.embedding.weight.numel()
conv_params = sum(p.numel() for name, p in model.named_parameters() if 'conv' in name)
fc_params = sum(p.numel() for name, p in model.named_parameters() if 'fc' in name or 'output' in name)

print(f"Model Parameter Breakdown:")
print(f"Total parameters: {total_params:,}")
print(f"Embedding parameters: {embedding_params:,} ({embedding_params/total_params*100:.1f}%)")
print(f"Convolutional parameters: {conv_params:,} ({conv_params/total_params*100:.1f}%)")
print(f"Fully connected parameters: {fc_params:,} ({fc_params/total_params*100:.1f}%)")

print(f"\nDataset Statistics:")
print(f"Training samples: {len(train_dataset)}")
print(f"Samples per parameter: {len(train_dataset)/total_params:.2f}")
print(f"K-mer vocabulary size: {train_dataset.vocab_size}")
print(f"Max sequence length: {train_dataset.max_length} k-mers")


## Summary and Conclusions


In [ ]:
print("=" * 80)
print("K-MER EMBEDDING CNN RESULTS SUMMARY")
print("=" * 80)
print(f"Architecture: K-mer Embedding CNN")
print(f"K-mer size: {K_MER_SIZE}")
print(f"Embedding dimension: 128")
print(f"Total parameters: {total_params:,}")
print(f"")
print(f"Training Results:")
print(f"  Best validation AUC: {best_val_auc:.4f}")
print(f"  Final validation accuracy: {history['val_accuracies'][-1]:.4f}")
print(f"")
print(f"Test Results:")
print(f"  Test accuracy: {test_acc:.4f}")
print(f"  Test AUC: {test_auc:.4f}")
print(f"  Test F1: {f1_score(test_targets, test_preds):.4f}")
print(f"")
print(f"Key Advantages of K-mer Embeddings:")
print(f"  ✓ Captures local sequence patterns effectively")
print(f"  ✓ Learned embeddings can represent k-mer relationships")
print(f"  ✓ More compact representation than one-hot encoding")
print(f"  ✓ Can handle variable-length sequences naturally")
print("=" * 80)

# Compare with target performance
target_accuracy = 0.93  # Target from literature
if test_acc >= target_accuracy:
    print(f"🎉 ACHIEVED TARGET: Test accuracy {test_acc:.4f} >= {target_accuracy:.2f}")
else:
    print(f"📈 PROGRESS: Test accuracy {test_acc:.4f}, target {target_accuracy:.2f} (gap: {target_accuracy - test_acc:.4f})")
    print(f"   Suggestions for improvement:")
    print(f"   - Try different k-mer sizes (4, 5, 7, 8)")
    print(f"   - Experiment with embedding dimensions")
    print(f"   - Add bidirectional LSTM layers")
    print(f"   - Use ensemble methods")
    print(f"   - Hyperparameter optimization")
